In [1]:
import pandas as pd
from rouge_score import rouge_scorer
from sarathi.benchmark.request_generator.real_request_generator import RealRequestGenerator
from sarathi.benchmark.config import Config

/workspace/miniconda3/envs/vattn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-05-12 02:59:29,457	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


unable to import module pod_attn


In [2]:
request_generator = RealRequestGenerator(config=Config({'num_requests': 100}))
cnn_prompts = request_generator.get_cnn_prompts()


In [8]:
scorer = rouge_scorer.RougeScorer(['rougeL', 'rouge1', 'rouge2'])

def get_rougeL_score(prompt, output):
    return scorer.score(prompt, output)['rougeL']

def get_avg_rougeL_score(prompts, outputs):
    precisions = []
    recalls = []
    fmeasures = []
    for prompt, output in zip(prompts, outputs):
        scores = get_rougeL_score(prompt, output)
        precisions.append(scores.precision)
        recalls.append(scores.recall)
        fmeasures.append(scores.fmeasure)
    return sum(precisions) / len(precisions), sum(recalls) / len(recalls), sum(fmeasures) / len(fmeasures)


In [15]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
precisions = []
recalls = []
fmeasures = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
precisions.append(avg_precision)
recalls.append(avg_recall)
fmeasures.append(avg_fmeasure)

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_fmeasure = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    precisions.append(avg_precision)
    recalls.append(avg_recall)
    fmeasures.append(avg_fmeasure)
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "precision": precisions,
    "recall": recalls,
    "fmeasure": fmeasures
})
df

,policy,throughput,precision,recall,fmeasure
0,ee_nobatch,26.282841,0.166531,0.143378,0.120703
1,off,71.546552,0.407214,0.241450,0.288385
2,eager,93.584302,0.025629,0.034102,0.028740
3,lazy,73.238791,0.382462,0.242704,0.279157
4,average,74.344407,0.359772,0.249517,0.279129
5,rebatching,80.815509,0.153599,0.112434,0.117615


In [9]:
policies = ["ee_nobatch", "off","eager", "lazy", "average", "rebatching"]
throughputs = []
rouge_scores = []

num_ee_tokens = []
num_no_ee_tokens = []
avg_conf_score = []
avg_conf_score_ee = []

ee_df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/ee_batch1.csv")
avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, ee_df.output)
throughputs.append(ee_df.throughput.mean())
rouge_scores.append(avg_rouge_score)

num_ee_tokens.append(ee_df.num_ee_tokens.values[0])
num_no_ee_tokens.append(ee_df.num_no_ee_tokens.values[0])
avg_conf_score.append(ee_df.avg_conf_score.values[0])
avg_conf_score_ee.append(ee_df.avg_conf_score_ee.values[0])

for policy in policies[1:]:
    df = pd.read_csv(f"/workspace/xutingl/vattention-ee/outputs_70b/req_100_batch_4_csv/{policy}.csv")
    avg_precision, avg_recall, avg_rouge_score = get_avg_rougeL_score(cnn_prompts, df.output)
    throughputs.append(df.throughput.mean())
    rouge_scores.append(avg_rouge_score)

    num_ee_tokens.append(df.num_ee_tokens.values[0])
    num_no_ee_tokens.append(df.num_no_ee_tokens.values[0])
    avg_conf_score.append(df.avg_conf_score.values[0])
    avg_conf_score_ee.append(df.avg_conf_score_ee.values[0])
df = pd.DataFrame({
    "policy": policies,
    "throughput": throughputs,
    "ROUGE-L": rouge_scores,
    "num_ee_tokens": num_ee_tokens,
    "num_no_ee_tokens": num_no_ee_tokens,
    "avg_conf_score": avg_conf_score,
    "avg_conf_score_ee": avg_conf_score_ee
})
df

,policy,throughput,ROUGE-L,num_ee_tokens,num_no_ee_tokens,avg_conf_score,avg_conf_score_ee
0,ee_nobatch,26.552540,0.120703,32306,7970,0.908029,0.985430
1,off,72.612111,0.288385,0,1,0.757795,0.000000
2,eager,94.451514,0.028740,25432,169,0.919896,0.925298
3,lazy,73.143975,0.279157,444,16681,0.732171,0.952019
4,average,75.331483,0.279129,2344,16529,0.707134,0.903871
5,rebatching,81.450985,0.117615,14756,5145,0.792330,0.982013
